# Topic Modelling with BERTopic


This notebook will guide you through producing a topic model using BERTopic together with a pre-trained transformer model.

> 💡 If you have brought your own dataset, try applying these steps to it at the end of this notebook. 

<img src="../../../../pictures/lego_stack/01_preprocessing.png" alt="Preprocessing diagram" style="max-width: 150px;">
<img src="../../../../pictures/lego_stack/03_vectorization.png" alt="Vectorization diagram" style="max-width: 150px;">
<img src="../../../../pictures/lego_stack/04_dimred_cluster.png" alt="Dimensionality Reduction and Clustering diagram" style="max-width: 150px;">
<img src="../../../../pictures/lego_stack/05_modelling.png" alt="Modelling diagram" style="max-width: 150px;">
<img src="../../../../pictures/lego_stack/06_evaluation.png" alt="Evaluation diagram" style="max-width: 150px;">

see https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html




<img src="../../../../pictures/bert_stack.svg" alt="Evaluation diagram" style="max-width: 400px;">

`c-TF-IDF` = "class‑based TF‑IDF". BERTopic builds one big pseudo‑document per topic (concatenating all documents assigned to that topic) and computes TF‑IDF where the IDF is computed across topics (classes) instead of across individual documents. This yields topic‑level term importance used to pick the top words that describe each topic.

## 0. Setup

In [1]:
#you will need the following packages installed to your python environment: 
#%pip install bertopic sentence-transformers transformers umap-learn HDBSCAN hf_xet nbformat 

#for loading the 20 newsgroups dataset:
from sklearn.datasets import fetch_20newsgroups

#import BERTopic and related packages:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer # for topic representation
from sentence_transformers import SentenceTransformer
from umap import UMAP #for dimensional reduction
from hdbscan import HDBSCAN #for clustering

/Users/rupertkiddle/opt/miniconda3/envs/gesis_iml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Preprocessing  
<img src="../../../../pictures/lego_stack/01_preprocessing.png" alt="Preprocessing diagram" style="max-width: 150px;">

In this step, we will load in the data, explore it, and clean it up a bit. 

We're working with the fetch_20newsgroups dataset from sklearn, which contains around 18,000 newsgroup posts on 20 topics.

In [ ]:
#load the 20 newsgroups dataset:
newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
docs = newsgroups.data

#drop any empty documents:
docs = [doc for doc in docs if len(doc.strip()) > 0]

#let's check the number of documents:
print (f"Number of documents: {len(docs)}")

print("First document:")
print(docs[0])

> 💡 That's it! Transformer models expect the text to be in 'natural' form - this means we do NOT perform any lowercasing, stopword removal, or lemmatization/stemming; since that would destroy the contextual information that the model relies on. 

# 2. Vectorization

<img src="../../../../pictures/lego_stack/03_vectorization.png" alt="Vectorization diagram" style="max-width: 150px;">

In this step, we're going to create embeddings of our documents using a pre-trained transformer model - `all-MiniLM-L6-v2` from the `sentence-transformers` library.

We will go into how these work on detail on D5 - they create a dense vector representation of the text that captures its overall semantic meaning.

> ⚠️ Most embedding models have a maximum 'context length' of 512 tokens. For certain kinds of texts, it is sufficient to embed only the first 512 tokens - for example, news articles, since the most important information is usually at the start. For other texts, you may want to 'chunk' the text into smaller pieces and embed each piece separately. 

In [ ]:
#create a list of corresponding document word lengths: 
doc_lengths = [len(doc.split()) for doc in docs]
print(f"Average document length (in words): {sum(doc_lengths)/len(doc_lengths):.2f}")
print(f"Maximum document length (in words): {max(doc_lengths)}")
print(f"Minimum document length (in words): {min(doc_lengths)}")

In [ ]:
# Define an embedding model to vectorize our documents:
# NOTE: cuda > cpu, or use "mps" for Apple Silicon
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="mps")

#precompute the embeddings (this may take a while on a CPU!):
embeddings = embedding_model.encode(docs, show_progress_bar=True)

# 3. Dimensionality Reduction and Clustering
<img src="../../../../pictures/lego_stack/04_dimred_cluster.png" alt="Dimensionality Reduction and Clustering diagram" style="max-width: 150px;">

In this step we do two things: 

1. Reduce the dimensionality of our embeddings, to avoid the 'curse of dimensionality' and make clustering more effective. We use `UMAP` for this, which is very good at preserving both global and local structure in the data.

2. Cluster the reduced embeddings using `HDBSCAN`, a density-based clustering algorithm that can find clusters of varying shapes and sizes, and can also identify outliers (documents that don't belong to any cluster). These clusters are our latent topics, which we will then interpret and refine descriptions of in the next step. 

In [ ]:
#The default dimensionality reduction algorithm in BERTopic is UMAP.
umap_model = UMAP(n_neighbors=15, n_components=5, 
                  min_dist=0.0, metric='cosine', random_state=42)

In [ ]:
#The default clustering algorithm in BERTopic is HDBSCAN.
#(A density-based clustering algorithm that can find clusters of varying shapes and sizes.
hdbscan_model = HDBSCAN(min_cluster_size=150, metric='euclidean', 
                        cluster_selection_method='eom', prediction_data=True)

# 4. Modelling
<img src="../../../../pictures/lego_stack/05_modelling.png" alt="Modelling diagram" style="max-width: 150px;">

In this step, we fit our BERTopic model to the documents, using the embedding model, UMAP model, and HDBSCAN model we defined earlier. This may take a while on a CPU!

In [ ]:
#we're also going to define a vectorizer, for topic representation:
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

In [ ]:
#train the BERTopic model, passing our embeddings, UMAP, and HDBSCAN models:

topic_model = BERTopic(embedding_model=embedding_model, #transformer model
                       umap_model=umap_model, #for dim. reduction
                       hdbscan_model=hdbscan_model, #for clustering
                       vectorizer_model=vectorizer_model, #for topic representation
                       calculate_probabilities=False) #for multiple topics (probs)

#bertopic returns topics, and probabilities for each document belonging to each topic
topics, probs = topic_model.fit_transform(docs, embeddings)

# 5. Evaluation
<img src="../../../../pictures/lego_stack/06_evaluation.png" alt="Evaluation diagram" style="max-width: 150px;">

In this step, we will explore the topics discovered by our BERTopic model, and consider fine-tuning the topic representations. 

In [ ]:
#let's see the topics:
topic_model.get_topic_info()

In [ ]:
#let's get the topN (c-TF-IDF based) words for a specific topic:
topic_model.get_topic(1)

In [ ]:
topic_model.visualize_barchart(top_n_topics=10)

In [ ]:
#visualize the hierarchy of topics:
topic_model.visualize_hierarchy()

In [ ]:
# Reduce dimensionality to 2D for visualization (now look at the simplified 2D representation of each document for illustration (rather than the 5D used as input for the clustering above))
umap_model_2 = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
reduced_embeddings = umap_model_2.fit_transform(embeddings)

# Now visualize how the documents are distributed in the 2D space (with topic labels per document based on the BERTopic model)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings)

In [ ]:
topic_model.visualize_heatmap()

## 6. Advanced: building your own BERTopic stack

### Modularity - any of the BERTopic components can be swapped out -

<img src="https://raw.githubusercontent.com/annekroon/gesis-machine-learning/main/pictures/modularity_bert.png" alt="Default Settings of BERTopic" width="600" height="400">

https://maartengr.github.io/BERTopic/algorithm/algorithm.html#visual-overview

Let's train another BERTopic model, this time including some steps to refine our topic representations: 


In [ ]:
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired

# Step 1 - Extract embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="mps") # Use "cuda" for GPU, "mps" for Apple Silicon, or "cpu"

# Step 2 - Reduce dimensionality
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

# Step 3 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=150, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Step 4 - Tokenize topics
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

# Step 5 - Create topic representations with c-TF-IDF
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True) # c-TF-IDF model (configurable)

# Step 6 - (Optional) Fine-tune topic representations with 

# Configure first refinement model (KeyBERT-inspired):
#NOTE: random n=500 docs from topic, take 10 most similar, then take top 100 words:
from bertopic.representation import KeyBERTInspired
keybert_model = KeyBERTInspired(top_n_words=100) 

# Configure second refinement model (MMR)
#NOTE: take top100 words, then apply MMR to select diverse top10 set:
from bertopic.representation import MaximalMarginalRelevance
mmr_model = MaximalMarginalRelevance(diversity=0.3, top_n_words=20)

# ...combine representation models:
representation_models = [keybert_model, mmr_model]

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_models, # Step 6 - (Optional) Fine-tune topic represenations
  calculate_probabilities=False               # Calculate probabilities (for multiple topics per document)
)


In [ ]:
topics, probs = topic_model.fit_transform(docs)

In [ ]:
topic_model.visualize_documents(docs, embeddings=embeddings)

In [ ]:
# Reduce dimensionality of embeddings, this step is optional but much faster to perform iteratively:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings)

In [ ]:
#The default dimensionality reduction algorithm in BERTopic is UMAP.
umap_model = UMAP(n_neighbors=15, n_components=5, 
                  min_dist=0.0, metric='cosine', random_state=42)
#The default clustering algorithm in BERTopic is HDBSCAN.
#(A density-based clustering algorithm that can find clusters of varying shapes and sizes.
hdbscan_model = HDBSCAN(min_cluster_size=150, metric='euclidean', 
                        cluster_selection_method='eom', prediction_data=True)

In [ ]:
topics, probs = topic_model.fit_transform(docs, embeddings)

## 7. Temporal Topic Modelling (advanced) - 

We can measure how topics change over time, if we have timestamps for our documents.

For a detailed explanation, see the BERTopic documentation:

https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html 

### get the CC news dataset from Huggingface - 


In [2]:
import random #for random sampling
from datasets import load_dataset #Hugging Face datasets library

#load the dataset in streaming mode (doesn't load everything into memory)
news_stream = load_dataset('cc_news', split='train', streaming=True)

#convert the stream to a list of 2000 random samples -
news_list = list(news_stream)
sampled_news = random.sample(news_list, 10000)

#print date and text of the first sampled article
print(sampled_news[0]['date'])
print(sampled_news[0]['text'])

2017-12-11 06:22:16
The Royal Thai Navy’s Mekong patrol unit has seized 610 kilograms of “premium gold” quality marijuana, the unit’s chief told a press conference on Monday.
Commander Adisak Phanurojanakarn said the marijuana was seized late on Friday night after the unit received tips-off that a trafficking ring would smuggle the drug across the Mekong River at Ban Chai Buri in Nakhon Phanom’s Tha Uthen distirct.
Full story: The Nation
By The Nation


### pre-process the data - we need lists of articles (strings) and their timestamps (datetime objects) -  


In [3]:
#get the texts and dates into separate lists:
docs = [article['text'] for article in sampled_news]
dates = [article['date'] for article in sampled_news]

#for the dates, we need to convert them to datetime objects:
from datetime import datetime
# keep only the YYYY-MM-DD part (ignore timestamps); set None for missing/empty dates
dates = [
    datetime.strptime(d.split()[0], "%Y-%m-%d") if isinstance(d, str) and d.strip() else None
    for d in dates
]

#iterate over docs and dates to remove any entries with None dates:
docs, dates = zip(*[(doc, date) for doc, date in zip(docs, dates) if date is not None])
docs, dates = list(docs), list(dates) #convert back to lists:


#validate input:
assert len(docs) == len(dates), "Documents and dates must have the same length"
assert all(isinstance(doc, str) and doc.strip() for doc in docs), "All documents must be non-empty strings"
assert all(d is not None for d in dates), "All dates must be valid datetime objects"


### configure our BERTopic stack - 


In [4]:
#define our BERTopic stack: 
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="mps") # Use "cuda" for GPU, "mps" for Apple Silicon, or "cpu"

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

hdbscan_model = HDBSCAN(min_cluster_size=20, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))  

temporal_topic_model = BERTopic(embedding_model=embedding_model, #transformer model
                       umap_model=umap_model, #for dim. reduction
                       hdbscan_model=hdbscan_model, #for clustering
                       vectorizer_model=vectorizer_model, #for topic representation
                       calculate_probabilities=False) #for multiple topics (probs)

### pre-compute embeddings to pass to the stack - 


In [5]:
#precompute the embeddings (this may take a while on a CPU!):
news_embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 278/278 [00:58<00:00,  4.77it/s]


### execute the stack (i.e., fit the model) - 

In [6]:
#fit our topic model:
temporal_topic_model.fit(docs, news_embeddings)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [7]:
#first, let's see the topics:
temporal_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3397,-1_said_people_year_new,"[said, people, year, new, time, like, years, p...",[Photo-Illustration: Getty Images\nWhen Donald...
1,0,483,0_shares_percent_stock_company,"[shares, percent, stock, company, market, pric...",[Rlj Lodging Trust (NYSE:RLJ) shares traded up...
2,1,386,1_league_club_season_united,"[league, club, season, united, premier league,...","[MANCHESTER, England, Aug 13 (Reuters) - The o..."
3,2,204,2_music_album_band_song,"[music, album, band, song, songs, singer, love...","[As a band member, every musician has a strong..."
4,3,179,3_dog_dogs_animal_animals,"[dog, dogs, animal, animals, said, deer, pet, ...",[Animal behaviour experts at the University of...
...,...,...,...,...,...
69,68,23,68_storm_hurricane_nate_gulf,"[storm, hurricane, nate, gulf, damage, irma, l...",[Deadly Tropical Storm Nate Threatens U.S. Gul...
70,69,23,69_uber_autonomous_vehicles_self driving,"[uber, autonomous, vehicles, self driving, veh...",[Pedestrian killed by self-driving Uber vehicl...
71,70,23,70_crashes_involving_distracted_road fatalities,"[crashes, involving, distracted, road fataliti...",[NO ONE has died on an upgraded stretch of the...
72,71,21,71_kim_jong_kim jong_korean,"[kim, jong, kim jong, korean, korea, north, no...",[The killing of the North Korean leader's outc...


### get temporally binned topic descriptions - 

In [8]:
#perform temporally binned C-TF-IDF to get topics over time:
topics_over_time = temporal_topic_model.topics_over_time(docs, dates, global_tuning=True, evolution_tuning=True, nr_bins=20)

In [11]:
# visualize the topics over time:
temporal_topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=20)

## 7. Semi-supervised Topic Modelling (advanced) -

We can nudge the topic model towards known topics by providing it with a set of seed documents for each topic.

This intervenes at the dimensionality reduction and clustering stage (i.e., foundational to the communities)

For a detailed explanation, see the BERTopic documentation:

https://maartengr.github.io/BERTopic/getting_started/semisupervised/semisupervised.html

In [12]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))
docs = data["data"]
categories = data["target"]
category_names = data["target_names"]

In [13]:
category_names

['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

In [14]:
#precompute the embeddings (this may take a while on a CPU!):
fetch20_embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 589/589 [02:01<00:00,  4.84it/s]


In [16]:
semi_topic_model = BERTopic(verbose=True).fit(docs, embeddings=fetch20_embeddings, y=categories)

2025-09-19 13:52:05,828 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-09-19 13:52:10,836 - BERTopic - Dimensionality - Completed ✓
2025-09-19 13:52:10,837 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been us

In [17]:
#let's see what we got: 
semi_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3884,-1_the_to_of_and,"[the, to, of, and, that, is, in, it, you, for]","[Tony-\n\nI read your post, it was nothing new..."
1,0,932,0_bike_dod_ride_my,"[bike, dod, ride, my, bikes, riding, motorcycl...",[\nIt depends on the bike. Once you've found a...
2,1,925,1_space_launch_nasa_orbit,"[space, launch, nasa, orbit, shuttle, mission,...",[Archive-name: space/controversy\nLast-modifie...
3,2,910,2_he_game_year_baseball,"[he, game, year, baseball, team, games, player...",[\nNot particularly *in* the World Series. Dur...
4,3,908,3_window_server_file_motif,"[window, server, file, motif, widget, entry, p...",[Archive-name: x-faq/speedups\nLast-modified: ...
...,...,...,...,...,...
184,183,11,183_ink_deskjet_printer_laser,"[ink, deskjet, printer, laser, printers, hp, i...",[\n\n\nI second that suggestion. Although I d...
185,184,11,184_voltage_generator_circuit_ac,"[voltage, generator, circuit, ac, 1a, output, ...","[Being in the ""visualization"" stage of a circu..."
186,185,11,185_turkey_truelove_leftover_christmas,"[turkey, truelove, leftover, christmas, served...","[\nJust taking a guess, perhaps it was that Ko..."
187,186,11,186_motto_things_worse_anthem,"[motto, things, worse, anthem, despise, harass...","[\n\nSo, we should ban the ammunition? Why no..."


In [22]:
semi_topic_model.visualize_topics()

In [26]:
semi_topic_model.visualize_hierarchy()